# Runner dos experimentos (Colab gratuito)

Roda os experimentos completos e grava em `MyDrive/litm-results`. **É retomável:** se a sessão cair ou a cota de GPU acabar, reabra este notebook (no dia seguinte, se precisar) e rode tudo de novo. Ele continua de onde parou.

1. **Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU**
2. **Ambiente de execução → Executar tudo**
3. Autorize o acesso ao Google Drive quando pedir.

In [1]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
assert gpu, "Sem GPU: Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU"
print("GPU:", gpu)

GPU: Tesla T4


In [3]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/litm-results"
REPO = "/content/lost-in-the-middle-attention"

Mounted at /content/drive


In [4]:
!test -d {REPO} || git clone -q https://github.com/henrique-voni/lost-in-the-middle-attention {REPO}
!git -C {REPO} pull -q
!pip install -q -r {REPO}/requirements.txt

## Progresso até agora

In [5]:
import os
import numpy as np
import pandas as pd

MODELS = ["phi4-mini", "smollm3-3b", "qwen3-4b"]

def status():
    for m in MODELS:
        a, c = f"{OUT}/attention_{m}.npz", f"{OUT}/accuracy_{m}.csv"
        att = int(np.load(a)["done"]) if os.path.exists(a) else 0
        acc = len(pd.read_csv(c)) if os.path.exists(c) else 0
        print(f"{m:11s} atenção {att:3d}/100 perguntas · acurácia {acc:4d}/1500 respostas")

status()

phi4-mini   atenção   0/100 perguntas · acurácia    0/1500 respostas
smollm3-3b  atenção   0/100 perguntas · acurácia    0/1500 respostas
qwen3-4b    atenção   0/100 perguntas · acurácia    0/1500 respostas


## Execução (atenção primeiro, que é mais barata; depois acurácia)

In [ ]:
%cd {REPO}
for exp, n in [("attention", 100), ("accuracy", 300)]:
    for m in MODELS:
        print(f"=== {m} · {exp} ===")
        !python scripts/run_experiments.py --model {m} --exp {exp} --n {n} --out {OUT}
status()

/content/lost-in-the-middle-attention
=== phi4-mini · attention ===
config.json: 100% 2.50k/2.50k [00:00<00:00, 8.83MB/s]
[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
tokenizer_config.json: 100% 2.93k/2.93k [00:00<00:00, 6.34MB/s]

tokenizer.json: downloading bytes:  17% 2.64M/15.5M [00:01<00:05, 2.31MB/s]
tokenizer.json: downloading bytes: 100% 5.00M/5.00M [00:01<00:00, 3.55MB/s,  482kB/s  ]
tokenizer.json: reconstructing file: 100% 15.5M/15.5M [00:01<00:00, 11.0MB/s, 1.50MB/s  ]
added_tokens.json: 100% 249/249 [00:00<00:00, 1.31MB/s]
special_tokens_map.json: 100% 587/587 [00:00<00:00, 3.14MB/s]
model.safetensors.index.json: 100% 16.3k/16.3k [00: